In [1]:
from utils import *
from plotly.subplots import make_subplots

results_dir = Path("../results/sanity_check")
scalars = TBScalars(".cache/sanity_check")

In [2]:
res_df = []
for test in results_dir.iterdir():
    env, seed = test.name.split("-")
    seed = int(seed.removeprefix("seed="))
    res_df.append({"path": test, "env": env, "seed": seed})
res_df = pd.DataFrame.from_records(res_df)
res_df

,path,env,seed
0,../results/sanity_check/Assault-seed=0,Assault,0
1,../results/sanity_check/CrazyClimber-seed=3,CrazyClimber,3
2,../results/sanity_check/CrazyClimber-seed=4,CrazyClimber,4
3,../results/sanity_check/Assault-seed=1,Assault,1
4,../results/sanity_check/CrazyClimber-seed=2,CrazyClimber,2
5,../results/sanity_check/CrazyClimber-seed=0,CrazyClimber,0
6,../results/sanity_check/Assault-seed=2,Assault,2
7,../results/sanity_check/MsPacman-seed=3,MsPacman,3
8,../results/sanity_check/MsPacman-seed=4,MsPacman,4
9,../results/sanity_check/Assault-seed=4,Assault,4


In [3]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    dfs = []
    for _, test in res_df[res_df["env"] == env].iterrows():
        df = scalars.read(test["path"])
        df = df[df["tag"] == "val/mean_ep_ret"]
        df["index"] = np.arange(len(df))
        dfs.append(df)
    df = pd.concat(dfs)

    g = df.groupby("index")
    avg_df = pd.DataFrame.from_records(
        {
            "score_mean": g["value"].mean(),
            "score_std": g["value"].std(),
            "step": g["step"].median(),
        }
    )

    color = next(make_color_iter())
    for trace in err_line(
        x=avg_df["step"],
        y=avg_df["score_mean"],
        std=avg_df["score_std"],
        color=color,
    ):
        fig.add_trace(trace, row=1, col=col)

fig

In [4]:
avg_df

,score_mean,score_std,step
index,,,
0,493.3125,43.827833,200003.0
1,375.3125,150.630997,450005.0
2,911.9375,256.161322,700008.0
3,994.8125,309.156536,950010.0
4,1136.8125,237.945963,1200011.0
5,1143.1875,127.756024,1450012.0
6,1454.7500,436.839200,1700014.0
7,1551.0625,349.691591,1950015.0
8,1549.5625,514.480482,2200016.0
